# **Computing Fourier Series**

<div style="color:#777777;margin-top: -15px;">
<b>Author</b>: Norman Juchler |
<b>Course</b>: ADLS ISP |
<b>Version</b>: v1.3 <br><br>
<!-- 05.03.2025: Partially refactored. -->
<!-- 20.02.2026: Partially refactored, split into two separate parts.. -->
</div>


Using *Fourier series*, we can approximate a periodic function $x(t)$ as a weighted sum of sinusoidal (= sine or cosine) basis functions. 

$$ x(t) \approx A_0 + \sum_{k=1}^N \left(A_k\cos(\omega_k t + \varphi_k) \right)$$

This decomposition allows us to analyze a signal in terms of its frequency components. Such a frequency-domain perspective is fundamental in many applications, including spectral filtering, image compression, and the solution of partial differential equations.

Note that the Fourier series is only one example of a series expansion. In a *Taylor series*, the basis functions are polynomials of increasing degree. In a *wavelet series*, the basis functions are wavelets – special functions that are localized in both time and frequency. We will explore what this localization means in practice later in the course.

For now, however, we will focus exclusively on the Fourier series.


## **Exercises**
* [Exercise 1 – Periodic signals](#exercise1)  
* [Exercise 2a – Compute Fourier series](#exercise2a)
* [Exercise 2b – Look up Fourier coefficients](#exercise2b)  
-->

---

## **Preparations**

In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from scipy import signal
from scipy.fft import fft, ifft, fftfreq, fftshift

# Jupyter / IPython configuration:
# Automatically reload modules when modified, if the extension is available
try:
    get_ipython().run_line_magic("load_ext", "autoreload")
    get_ipython().run_line_magic("autoreload", "2")
except Exception:
    pass

# Enable vectorized output (for nicer plots)
%config InlineBackend.figure_formats = ["svg"]

import sys
sys.path.insert(0, "../")
import isp
isp.setup_plotting()

---

<a id='exercise1'></a>

## **&#9734;  Exercise 1 – Periodic signals**

In the lectures, we have explored various periodic functions $x(t)$, such as the sawtooth and pulse functions.

The [`scipy.signal`](https://docs.scipy.org/doc/scipy/reference/signal.html#waveforms) module provides already built-in support for generating several types of waveforms. 
 

In [ ]:
# Illustration of different waveforms offered by numpy and scipy.signal:
duration = 1    # seconds
f_signal = 5    # Hz
f_sample = 1000 # Hz

t = np.arange(0, duration, 1/f_sample)

# Sine wave
x_sine = np.sin(2 * np.pi * f_signal * t)
x_rect = signal.square(2 * np.pi * f_signal * t, duty=0.3)
x_sawtooth = signal.sawtooth(2 * np.pi * f_signal * t)
x_triangle = signal.sawtooth(2 * np.pi * f_signal * t, width=0.5)

# Decorate the plot
def decorate(ax):
    ax.set_ylabel("x(t)")
    # Add legend outside of the plot
    ax.legend(loc="upper left", bbox_to_anchor=(1, 1.05))
    # Add a zero line
    ax.plot([0, duration], [0, 0], "k--", lw=0.5, alpha=0.5)

# Plot
fig, axes = plt.subplots(4, 1, figsize=(6, 6), sharex=True)
axes[0].plot(t, x_sine, label="Sine")
axes[1].plot(t, x_rect, label="Rectangular")
axes[2].plot(t, x_sawtooth, label="Sawtooth")
axes[3].plot(t, x_triangle, label="Triangle")
for ax in axes:
    decorate(ax)
# Only for last ax:
ax.set_xlabel("Time [s]");

Your task is to construct these signals using only NumPy or basic Python operations.

### **Instructions**
* Complete the following functions.
* Visualize the results to verify correctness.
* Hints:
  * The modulo operation `t%T` computes the remainder when dividing `t` by `T`. This can also be used for an array of values `t`.
  * The modulo operation `t%T` computes the remainder of `t` divided by `T`. When applied to an array `t`, it operates element-wise.
  * [`np.where(cond, x, y)`](https://numpy.org/doc/stable/reference/generated/numpy.where.html) assigns a value `x` if condition `cond` is True, and `y` otherwise.

In [ ]:
######################
###    EXCERISE    ###
######################

def rectangular_wave(t, f, duty=0.5):
    """Generate a rectangular wave.

    Parameters
    ----------
    t : array_like      Time vector
    f : float           Frequency of the wave
    duty : float        Duty cycle of the wave
    """
    ...


def sawtooth_wave(t, f):
    """Generate a sawtooth wave.

    Parameters
    ----------
    t : array_like      Time vector
    f : float           Frequency of the wave
    """
    ...


def triangle_wave(t, f):
    """Generate a triangle wave.

    Parameters
    ----------
    t : array_like      Time vector
    f : float           Frequency of the wave
    """
    ...


# Plot
fig, axes = plt.subplots(3, 1, figsize=(6, 6), sharex=True)
axes[0].plot(t, rectangular_wave(t, f_signal, duty=0.3), label="Rectangular")
axes[1].plot(t, sawtooth_wave(t, f_signal), label="Sawtooth")
axes[2].plot(t, triangle_wave(t, f_signal), label="Triangle")
for ax in axes:
    decorate(ax)
# Only for last ax:
ax.set_xlabel("Time [s]");

In [ ]:
######################
###    SOLUTION    ###
######################

def rectangular_wave(t, f, duty=0.5, amplitude=1, offset=0.):
    """Generate a rectangular wave.

    Parameters
    ----------
    t : array_like      Time vector
    f : float           Frequency of the wave
    duty : float        Duty cycle of the wave
    amplitude : float   Amplitude of the wave
    """
    t_period = 1 / f
    t_duty = t_period * duty
    return np.where((t % t_period) < t_duty, 1, -1) * amplitude + offset


def sawtooth_wave(t, f, amplitude=1, offset=0):
    """Generate a sawtooth wave.

    Parameters
    ----------
    t : array_like      Time vector
    f : float           Frequency of the wave
    amplitude : float   Amplitude of the wave
    """
    t_period = 1 / f
    return 2 * (t % t_period) / t_period * amplitude - amplitude + offset


def triangle_wave(t, f, amplitude=1, offset=0):
    """Generate a triangle wave.

    Parameters
    ----------
    t : array_like      Time vector
    f : float           Frequency of the wave
    amplitude : float   Amplitude of the wave
    """
    t_period = 1 / f
    return -2 * np.abs(2 * (t % t_period) / t_period - 1) * amplitude + amplitude + offset


t = np.arange(0, duration, 1/f_sample)

# Plot
fig, axes = plt.subplots(3, 1, figsize=(6, 6), sharex=True)
axes[0].plot(t, rectangular_wave(t, f_signal, duty=0.3), label="Rectangular")
axes[1].plot(t, sawtooth_wave(t, f_signal), label="Sawtooth")
axes[1].plot(t, sawtooth_wave(t, f_signal, offset=0.5, amplitude=0.5), label="Sawtooth (Ex. 2a)")
axes[2].plot(t, triangle_wave(t, f_signal), label="Triangle")
for ax in axes:
    decorate(ax)
# Only for last ax:
ax.set_xlabel("Time [s]");

---

<a id='exercise2a'></a>

## **&#9734;&#9734;  Exercise 2a – Compute Fourier series**

In the lecture, we introduced the procedure for calculating the Fourier coefficients. The following formulas are fundamental:

$$\begin{align*} 
a_0&= \frac{1}{T}\int_{-T/2}^{T/2} x(t)\,dt\\
a_k &= \frac{2}{T}\int_{-T/2}^{T/2} x(t) \cos \left( 2\pi \tfrac{k}{T} t \right) \,dt\qquad\text{for } k\geq 1\qquad\\
b_k &= \frac{2}{T}\int_{-T/2}^{T/2} x(t) \sin \left( 2\pi \tfrac{k}{T} t \right) dt\qquad\text{for } k\geq 1
\end{align*}
$$

These integrals yield the Fourier series coefficients (in sine-cosine form), which describe the signal in terms of its frequency components.

### **Example: Fourier Series of a square wave**
Let's apply these formulas to a square wave (rectangular wave with duty cycle = 0.5)
Since the Fourier integrals are evaluated over one period only, we define:

$$
x(t) = \begin{cases} -1 & \text{if } &-\frac{T}{2} \leq t < 0 
\\ +1 & \text{if }  &0 \leq t < \frac{T}{2} \end{cases} $$

It is important to note that the specific choice of the integration interval does not affect the result. For example, we could equivalently integrate over $0 \leq t \leq T$.


#### **Coefficient $a_0$ (DC component)**

$$
\begin{align*}
a_0 &= \frac{1}{T}\left(\int_{-T/2}^0 (-1)\,dt + \int^{T/2}_0 (+1)\,dt\right)\\
&= \frac{1}{T}\left(-t\,\bigg\rvert_{-T/2}^0 + t\,\bigg\rvert_{0}^{T/2}\right)\\
&= 0
\end{align*}
$$

The coefficient $a_0$ represents the DC component, i.e., the average value of the signal over one period. Since the square wave is symmetric around zero, its average value is indeed zero.


#### **Coefficient $a_k$ (cosine coefficients)**

$$
\begin{align*}
a_k &= \frac{2}{T}\left(\int_{-T/2}^0 (-1)\cos\left(\tfrac{2\pi k}{T}t\right) \,dt + \int^{T/2}_0 (+1)\cos\left(\tfrac{2\pi k}{T}t\right)\,dt\right)\\
&= -\frac{2}{T}\frac{T}{2\pi k}\sin\left(\frac{2\pi k}{T}t\right)\,\bigg\rvert_{-T/2}^0 + 
\frac{2}{T}\frac{T}{2\pi k}\sin\left(\frac{2\pi k}{T}t\right)\,\bigg\rvert_{-T/2}^0\\
 &= 0
\end{align*}
$$

All coefficients $a_k$ vanish. This follows directly from symmetry: The square wave defined above is an odd function, while cosine is an even function. The product of an odd and an even function is odd, and the integral of an odd function over a symmetric interval is zero.

Thus, no cosine terms contribute to the Fourier series.

#### **Coefficient $b_k$ (sine coefficients)**


$$
\begin{align*}
b_k &= \frac{2}{T}\left(\int_{-T/2}^0 (-1)\sin\left(\tfrac{2\pi k}{T}t\right) \,dt + \int^{T/2}_0 (+1)\sin\left(\tfrac{2\pi k}{T}t\right)\,dt\right) \\
&= \frac{2}{T}\frac{T}{2\pi k}\cos\left(\frac{2\pi k}{T}t\right)\,\bigg\rvert_{-T/2}^0 
- \frac{2}{T}\frac{T}{2\pi k}\cos\left(\frac{2\pi k}{T}t\right)\,\bigg\rvert_0^{T/2} \\[1em]
&= \frac{1}{\pi k}(\cos(0)- \cos(-\pi k)) - \frac{1}{\pi k}(\cos(\pi k)- \cos(0))\\[0.75em]
&= \frac{1}{\pi k}(1 - \cos(+\pi k)) - \frac{1}{\pi k}(\cos(\pi k)- 1)\\[0.75em]
&= \begin{cases} 0, & k\text{ even}\\
\frac{4}{\pi k}, & k\text{ odd}
\end{cases}
\end{align*}
$$

In the final step, we used:
* $\cos(-t)=\cos(t)$
* $\cos({\pi k})=(-1)^k$ 

Hence, only the odd harmonics (sine terms) contribute to the Fourier series.

<br>

### **Instructions**
* Carefully follow the above derivation and understand why symmetry arguments simplify the computation.
* For the ambitious: Repeat the calculation for a sawtooth wave and compare the harmonic structure.



```python
######################
###    EXCERISE    ###
######################
````

Solve this task with paper and pencil (or a tablet).
<br><br><br><br><br>




```python
######################
###    SOLUTION    ###
######################
````

### **Sawtooth wave**

We first derive the Fourier series for a specific sawtooth function with parameters $A=0.5$, $q=0.5$, and $T>0$ (as introduced in Exercise 1). Then, we generalize the results for arbitrary parameters $A>0$, $T>0$, and $q\geq 0$. Finally, we verify the solution by reconstructing the signal from its Fourier series.

The sawtooth function can be defined as follows:

$$
x_{saw}(t) = \dfrac{2At}{T} + q - A,\quad 0 \leq t < T\\[1em]
$$

For the particular choice of $A=\tfrac{1}{2}$, $q=\tfrac{1}{2}$ we get:
$$
x_{saw}(t) = \dfrac{t}{T},\quad 0 \leq t < T\\[1em]
$$

The parameter $A$ determines the amplitude of the signal, while $q$ controls its vertical offset. Let's now consider two different sets of parameter values and examine how they affect the resulting periodic signal.


In [ ]:
def sawtooth_wave(t, f, amplitude=1, offset=0):
    t_period = 1 / f
    return 2 * (t % t_period) / t_period * amplitude - amplitude + offset

f_sample = 1000 # Hz
t = np.arange(-duration/2, duration/2, 1/f_sample)

A = 1; q = 0; T=1/5
x0_saw = sawtooth_wave(t, f=1/T, amplitude=A, offset=q)

A = 0.5; q = 0.5; T = 1/5
x1_saw = sawtooth_wave(t, f=1/T, amplitude=A, offset=q)

A = -0.25; q = 2; T=1/5
x2_saw = sawtooth_wave(t, f=1/T, amplitude=A, offset=q)


fig, ax = plt.subplots(figsize=(6, 3))
ax.plot(t, x0_saw, label="Sawtooth 0 (A=1,     q=0,    T=1/5)")
ax.plot(t, x1_saw, label="Sawtooth 1 (A=1/2,  q=1/2, T=1/5)")
ax.plot(t, x2_saw, label="Sawtooth 2 (A=-1/4, q=2,    T=1/5)")
ax.set_ylabel("x(t)")
ax.set_xlabel("Time [s]")
ax.plot([-duration/2, duration/2], [0, 0], "--", color="k", lw=2)
ax.plot([0, 0], ax.get_ylim(), "--", color="k", lw=2)
ax.legend(loc="upper left", bbox_to_anchor=(1, 1.035))
ax.yaxis.grid(True, linestyle="--", alpha=0.5)
ax.set_axisbelow(True)

Since the function is periodic with period $T$, we can integrate over any interval of width $T$ to compute the coefficient of the Fourier series. Choosing $ 0\leq t \leq T$, the Fourier coefficients are given by:

$$\begin{array}{lll}
a_0 &= \displaystyle\frac{1}{T}\int_{0}^{T} \frac{t}{T}\,dt&\\
a_k &= \displaystyle\frac{2}{T}\int_{0}^{T} \frac{t}{T} \cos \left( 2\pi \tfrac{k}{T} t \right) \,dt &\text{for } k\geq 1\qquad\\
b_k &= \displaystyle\frac{2}{T}\int_{0}^{T}  \frac{t}{T} \sin \left( 2\pi \tfrac{k}{T} t \right)\,dt &\text{for } k\geq 1
\end{array}
$$

#### **Coefficient $a_0$**

The DC component $a_0$ can be calculated as:
$$
\begin{align*}
a_0 = \frac{1}{T}\int_0^T\frac{t}{T}\,dt = \frac{1}{2}\frac{t^2}{T^2}\;\,\bigg\rvert^{T}_0 = \frac{1}{2}
\end{align*}
$$

#### **Coefficient $a_k$ (cosine coefficients)**

Observing the graph of the sawtooth function, we can see that it is an *odd* function when centered around its DC component. Consequently, all cosine coefficients vanish, meaning $a_k=0$ for all $k$. You can verify this by directly evaluating the formula for $a_k$​ above.


#### **Coefficient $b_k$ (sine coefficients)**

The computation of $b_k$ is slightly more involved, as it requires **integration by parts** (German: *partielle Integration*).
Integration by parts is useful when the integrand can be written as the product of two functions, $u(t)$ and $v'(t)$. The formula reads:

$$
\int_D (uv')\,dt = (uv)\Big\rvert_{D} - \int_D (u'v)\,dt
$$

We need to evaluate the following definite integral:

$$
b_k=\frac{2}{T}\int_0^T\frac{t}{T}\sin\left(2\pi\tfrac{k}{T}t\right)\,dt\\[2em]
$$

We choose:

$$
\begin{align*}
& \quad\Rightarrow \quad u(t) = \tfrac{t}{T},\quad v'(t)=\sin\left(2\pi\tfrac{k}{T}t\right)\\
& \quad\Rightarrow \quad u'(t) = \tfrac{1}{T},\quad v(t)=\tfrac{-T}{2\pi k}\cos\left(2\pi\tfrac{k}{T}t\right)\\[1.5em]
\end{align*}
$$

Applying integration by parts:

$$
\begin{align*}
b_k=&\frac{2}{T}\left(-\frac{t}{T}\frac{T}{2\pi k}\cos\left(2\pi\tfrac{k}{T}t\right)\right)\bigg\rvert^T_0 
- \frac{2}{T}\frac{1}{T}\frac{-T}{2\pi k}\int_0^T \cos\left(2\pi\tfrac{k}{T}t\right)\,dt\\
=&\frac{-1}{\pi k} \cos\left(2\pi k\right) + \frac{1}{\pi k T}\left(\frac{T}{2\pi k}\sin\left(2\pi\tfrac{k}{T}t\right)\right)\bigg\rvert_0^T\\[2em]
\end{align*}
$$

Using trigonometric identities,
$$
\begin{align*}
\quad\Rightarrow \quad \cos(2\pi k) = 1\quad \forall k>0\\
\quad\Rightarrow \quad \sin(2\pi k) = 0\quad \forall k>0\\[1.5em]
\end{align*}
$$

...the sine term vanishes, and we obtain the final result:

$$
b_k =-\frac{1}{\pi k}
$$

<br><br>

#### **Generalization**

For an arbitrary sawtooth wave with parameters   the Fourier coefficients are:
For an arbitrary sawtooth wave with parameters $A>0, T>0$ and $q\geq0$, the Fourier coefficients are given by:

$$
\begin{align*}
a_0&= q\\
a_k&= 0\\
b_k&= -\tfrac{2A}{\pi k}
\end{align*}
$$

Thus, the signal consists of a constant offset $q$ and sine terms whose amplitudes decay proportionally to 
$\tfrac{1}{k}$

#### **Verification**

We can verify these results by reconstructing the sawtooth wave from its Fourier series using $N=25$ terms and comparing the approximation to the original signal.

In [ ]:
def plot_reconstruction(t, x_orig1, x_rec1, label_orig1, label_rec1,
                        x_orig2, x_rec2, label_orig2, label_rec2):
    """
    This is just a utility function to create the same type of plot.
    You do not have to understand the details of this function.
    """
    fig, ax = plt.subplots(figsize=(7, 4), sharex=False)
    ax.plot(t, x_rec1, label=label_rec1, 
            color=isp.PALETTE[0])
    ax.plot(t, x_orig1, label=label_orig1, 
            color=isp.PALETTE[1])
    ax.plot(t, x_rec2, label=label_rec2,
            color=isp.PALETTE[0], linestyle=":") 
    ax.plot(t, x_orig2, label=label_orig2, 
            color=isp.PALETTE[1], linestyle=":")
    ax.grid("on")
    ax.legend(loc="upper left", bbox_to_anchor=(1, 1.025))
    return fig, ax


def compute_reconstruction(t, T, a0, aks, bks):
    """
    This function reconstructs a periodic signal from its Fourier 
    coefficients a0, aks, and bks. T is the period of the signal
    and t is the time vector.
    """
    assert len(aks) == len(bks)
    n = len(aks)
    x_rec = np.zeros_like(t)
    x_rec += a0
    for k in range(1, n):
        x_rec += bks[k-1] * np.sin(2 * np.pi * k / T * t)
        x_rec += aks[k-1] * np.cos(2 * np.pi * k / T * t)
    return x_rec

# Settings that we will use for all reconstructions.
duration = 1    # seconds
f_signal = 5    # Hz
f_sample = 1000 # Hz
t = np.arange(0, duration, 1/f_sample)
N = 25

In [ ]:
####################
# Sawtooth signal 
####################

def compute_coefficients(A, q, N):
    a0 = q
    aks = [0                for k in range(1, N+1)]
    bks = [-2*A/(np.pi * k) for k in range(1, N+1)]
    return a0, aks, bks


# Sawtooth 1:
A = 0.5
q = 0.5
T = 1/f_signal

a0, aks, bks = compute_coefficients(A, q, N)
x_orig1 = sawtooth_wave(t, f_signal, offset=q, amplitude=A)
x_rec1 = compute_reconstruction(t, T, a0, aks, bks)

# Sawtooth 2: 
A = -0.25
q = 2
T = 1/f_signal

a0, aks, bks = compute_coefficients(A, q, N)
x_orig2 = sawtooth_wave(t, f_signal, offset=q, amplitude=A)
x_rec2 = compute_reconstruction(t, T, a0, aks, bks)

plot_reconstruction(t, 
                    x_orig1=x_orig1, 
                    x_rec1=x_rec1, 
                    label_orig1=r"Sawtooth 1: $x(t)=\frac{t}{T}$",
                    label_rec1=r"Sawtooth 1: reconstructed",
                    x_orig2=x_orig2,
                    x_rec2=x_rec2,
                    label_orig2=r"Sawtooth 2 $x(t)=\frac{2At}{T}+q-A$",
                    label_rec2=r"Sawtooth 2: reconstructed");

Note the ripples near the discontinuities. Although the overall approximation improves as more terms are added, these oscillations persist and never fully disappear. This effect is known as the [**Gibbs phenomenon**](https://en.wikipedia.org/wiki/Gibbs_phenomenon) and occurs in the Fourier series of functions with jump discontinuities, such as the sawtooth wave. It arises from the overshooting of the sinusoidal components near the points of discontinuity. Remarkably, this overshoot does not vanish as $N\rightarrow\infty$; it approaches a constant value of approximately 9% of the jump height.


---

### **Triangular wave**

The triangular wave is defined as

$$x(t)=
\begin{cases}
\begin{array}{lrl}
-\frac{4A}{T}t - A + q,\quad &-T/2 &\leq t < 0\\
\frac{4A}{T}t - A + q,\quad &0 &\leq t < T/2
\end{array}
\end{cases}
$$

The corresponding Fourier series coefficients are obtained by evaluating the defining integrals. The result is:

$$
\begin{array}{lll}
a_0 &= A+q\\
a_k &=-\frac{8A}{\pi^2k^2}\\
b_k &= 0
\end{array}
$$

Thus, the triangular wave consists solely of cosine terms whose amplitudes decay proportionally to 
$\tfrac{1}{k^2}, indicating significantly faster convergence compared to the sawtooth wave.

Let's visualize the results.

In [ ]:
####################
# Triangular signal
####################

def compute_coefficients(A, q, N):
    a0 = q
    aks = [- 8*A/(np.pi**2 * k**2) * (k%2) for k in range(1, N+1)]
    bks = [0                               for k in range(1, N+1)]
    return a0, aks, bks


# Sawtooth 1:
A = 0.5
q = 0.5
T = 1/f_signal

a0, aks, bks = compute_coefficients(A, q, N)
x_orig1 = triangle_wave(t, f_signal, offset=q, amplitude=A)
x_rec1 = compute_reconstruction(t, T, a0, aks, bks)

# Sawtooth 2: 
A = -0.25
q = 2
T = 1/f_signal

a0, aks, bks = compute_coefficients(A, q, N)
x_orig2 = triangle_wave(t, f_signal, offset=q, amplitude=A)
x_rec2 = compute_reconstruction(t, T, a0, aks, bks)

plot_reconstruction(t, 
                    x_orig1=x_orig1, 
                    x_rec1=x_rec1, 
                    label_orig1="Triangle 1: original",
                    label_rec1="Triangle 1: reconstructed",
                    x_orig2=x_orig2,
                    x_rec2=x_rec2,
                    label_orig2="Triangle 2: original",
                    label_rec2="Triangle 2: reconstructed");

---

### **Rectangular wave**

The generalized square wave function or rectangular wave function is defined as

$$x(t)=
\begin{cases}
\begin{array}{lrl}
A+q,\quad &0 &\leq t < r\cdot T\\
-A+q,\quad &r\cdot T &\leq t < T\\
\end{array}
\end{cases}
$$

where $r\in(0,1)$ denotes the duty cycle.

By evaluating the corresponding integrals, we obtain the Fourier series coefficients:

$$
\begin{align*}
a_0 &= A\cdot(2\cdot r-1)+q\\
a_k &= \frac{2A}{\pi k}\sin(2\pi k r)\\
b_k &= \frac{2A}{\pi k}\left(1-\cos(2\pi k r)\right)
\end{align*}
$$

Again, let's examine how these coefficients determine the shape of the reconstructed signal.

In [ ]:
####################
# Rectangular signal
####################

def compute_coefficients(A, q, r, N):
    a0 = A*(2*r-1)+q
    aks = [2*A/(np.pi*k)*np.sin(2*np.pi*k*r) for k in range(1, N+1)]
    bks = [2*A/(np.pi*k)*(1-np.cos(2*np.pi*k*r)) for k in range(1, N+1)]
    return a0, aks, bks


# Rectangular 1:
r = 0.5
A = 0.5
q = 0.5
T = 1/f_signal

a0, aks, bks = compute_coefficients(A, q, r, N)
x_orig1 = rectangular_wave(t, f_signal, duty=r, amplitude=A, offset=q)
x_rec1 = compute_reconstruction(t, T, a0, aks, bks)

# Rectangular 2: 
r = 0.25
A = -0.4
q = 2.3
T = 1/f_signal

a0, aks, bks = compute_coefficients(A, q, r, N)
x_orig2 = rectangular_wave(t, f_signal, duty=r, amplitude=A, offset=q)
x_rec2 = compute_reconstruction(t, T, a0, aks, bks)

plot_reconstruction(t, 
                    x_orig1=x_orig1, 
                    x_rec1=x_rec1, 
                    label_orig1="Rectangular 1: original",
                    label_rec1="Rectangular 1: reconstructed",
                    x_orig2=x_orig2,
                    x_rec2=x_rec2,
                    label_orig2="Rectangular 2: original",
                    label_rec2="Rectangular 2: reconstructed");


---

<a id='exercise2b'></a>

## **&#9734;  Exercise 2b – Look up Fourier coefficients**

Alternatively, instead of deriving the Fourier coefficients manually, we can consult a formulary and look up the coefficients for specific functions $x(t)$

### **Instructions**
* Search the web for the Fourier series coefficients of:
  * the *rectangular wave*, 
  * the *sawtooth wave*, and 
  * the *triangular wave*.
* Use the *sine-cosine form* of the Fourier series!
* Carefully ensure that the referenced formulas correspond exactly to the definition of $x(t)$ introduced above.


```python
######################
###    EXCERISE    ###
######################
````

Type here your solution.


***Formulas for rectangular wave***

$a_0 = ... $  
$a_k = ... $  
$b_k = ... $  

***Formulas for sawtooth wave***

$a_0 = ... $  
$a_k = ... $  
$b_k = ... $  

***Formulas for traingular wave***

$a_0 = ... $  
$a_k = ... $  
$b_k = ... $  

<br><br><br>



```python
######################
###    SOLUTION    ###
######################
````

Compare your results with [Exercise 2a](#exercise2a). If your Fourier coefficients differ, first verify that you used exactly the same function definition. In particular, check the amplitude, period, offset, and (if applicable) duty cycle. Also note that the solution presented here uses parameterized versions of the waveforms. If you chose different parameter values, your coefficients will differ accordingly.

<br>

***Formulas for rectangular wave***

$
\begin{aligned}
a_0 &= A\cdot(2\cdot r-1)+q\\[0.3em]
a_k &= \tfrac{2A}{\pi k}\sin(2\pi k r)\\[0.3em]
b_k &= \tfrac{2A}{\pi k}\left(1-\cos(2\pi k r)\right)
\end{aligned}
$

<br>

***Formulas for sawtooth wave***

$
\begin{aligned}
a_0 &= q\\[0.3em]
a_k &= 0\\[0.2em]
b_k &= -\tfrac{2A}{\pi k}
\end{aligned}
$

<br>

***Formulas for traingular wave***


$
\begin{aligned}
a_0 &= A+q\\[0.3em]
a_k &= -\tfrac{8A}{\pi^2k^2}\\[0.3em]
b_k &= 0
\end{aligned}
$